In [13]:
%%capture
%pip install torch transformers datasets peft trl bitsandbytes accelerate


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [36]:
from datasets import load_dataset
import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, DataCollatorForLanguageModeling, Trainer, pipeline
import gc
import torch
from peft import LoraConfig, get_peft_model

In [28]:
# 1. Load raw jsonl
data = load_dataset("json", data_files="../../Data/Finetuning/synthetic/Small/Llama/dolly_train_all_Llama.jsonl", split="train")


In [29]:
SYSTEM_PROMPT = "You are a helpful assistant."

def build_prompt(example):
    prompt = f"{SYSTEM_PROMPT}\nInstruction: {example['instruction']}\nResponse:"
    example["text"] = prompt + " " + example["response_model"]
    return example

data = data.map(build_prompt)


In [41]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct") 

# Add special tokens if needed
tokenizer.add_special_tokens({"pad_token": "[PAD]"})

def preprocess(ex):
    toks = tokenizer(
        ex["text"],
        truncation=False,
        max_length=2048,
        padding="max_length"
    )
    # for causal LM, labels = input_ids (pads → –100 will be set by collator or manually)
    toks["labels"] = toks["input_ids"].copy()
    return toks

def filter_by_length(example):
    length = len(tokenizer(example["text"], truncation=False).input_ids)
    return length <= 2048

data = data.filter(filter_by_length, batched=False)

tokenized = data.map(preprocess, batched=True, remove_columns=data.column_names)


Map: 100%|██████████| 13480/13480 [00:09<00:00, 1455.39 examples/s]


In [50]:
lora_config = LoraConfig(
    r=16,                        # rank of LoRA update matrices
    lora_alpha=32,               # scaling
    target_modules=["q_proj","v_proj"],  
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [32]:
base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    device_map="auto",
    torch_dtype="float16",
    trust_remote_code=True
)

model = get_peft_model(base_model, lora_config)

/home/mschaffeld/venv_310/lib/python3.10/site-packages/accelerate/utils/modeling.py:1569: UserWarning: Current model requires 128 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.46it/s]


In [51]:
training_args = TrainingArguments(
    output_dir="../../Data/ft_models/lora-llama3.1-8b",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_steps=2000,
    learning_rate=3e-4,
    logging_steps=50,
    save_steps=500,
    fp16=True,
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    label_names=["labels"]
)

In [52]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer,
    mlm=False,
    pad_to_multiple_of=8
)

In [53]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=data,
    data_collator=data_collator
)

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [54]:
trainer.train()
model.save_pretrained("lora-llama3.1-8b-finetuned")


ValueError: No columns in the dataset match the model's forward method signature. The following columns have been ignored: [text, response_human, model_name, instruction, response_model, index, category]. Please check the dataset and model. You may need to set `remove_unused_columns=False` in `TrainingArguments`.

In [56]:
# 1. Load & format
from datasets import load_dataset
data = load_dataset("json", data_files="../../Data/Finetuning/synthetic/Small/Llama/dolly_train_all_Llama.jsonl", split="train")

# 2. Build “text” and tokenize+label
data = data.map(lambda ex: {
    "text": "You are a helpful assistant.\nInstruction: " 
            + ex["instruction"] 
            + "\nResponse: " 
            + ex["response_model"]
}, remove_columns=data.column_names)

# 3. Tokenize & set labels
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", use_fast=True)
tokenizer.add_special_tokens({"pad_token":"[PAD]"})
def tok_and_label(ex):
    toks = tokenizer(ex["text"], truncation=True, max_length=2048, padding="max_length")
    toks["labels"] = toks["input_ids"].copy()
    return toks

tokenized = data.map(tok_and_label, batched=True, remove_columns=["text"])

# 4. Trainer w/ correct columns
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False, pad_to_multiple_of=8)

training_args = TrainingArguments(
  output_dir="out",
  per_device_train_batch_size=2,
  gradient_accumulation_steps=8,
  fp16=True,
  remove_unused_columns=False,    # optional if you removed raw columns
  label_names=["labels"],         # ensure Trainer knows your label key
)

trainer = Trainer(
  model=model,
  args=training_args,
  train_dataset=tokenized,
  data_collator=data_collator,
)

trainer.train()


Map: 100%|██████████| 13509/13509 [00:09<00:00, 1450.43 examples/s]
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [80,0,0], thread: [0,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [80,0,0], thread: [1,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [80,0,0], thread: [2,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [80,0,0], thread: [3,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [80,0,0], thread: [4,0,0] Asser

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
